# Retail demand forecasting
Reproduce the Friday-aligned model comparison, missing-data audit, and 26-week Prophet forecasts. Run all cells with the repository root as the kernel working directory and the pinned forecasting environment active.

The benchmark covers 164 consistently observed series. The full-panel sensitivity uses cutoff-safe training fills and scores only Fridays with all 168 actuals observed. See the [validation report](docs/forecasting-validation.md) for the model settings, split, and limitations. The [original notebook](historical/Demand_forecasting.ipynb) contains the submitted exploratory analysis and saved historical outputs.

In [ ]:
from pathlib import Path

from forecasting.evaluate import evaluate

root = Path.cwd()
results = evaluate(root, root / "results")
results["metrics"].round(2)

## Missing weeks and backtesting
Inspect the source gaps and the score for each forecast origin.

In [ ]:
results["missing_gaps"]

In [ ]:
results["fold_metrics"].query(
    "scope == 'complete_164' and evaluation == 'rolling'"
).round(2)

## Holdout and 26-week forecast
Compare the observed 164-series holdout against each model. The final panel shows Prophet's nominal 80% model interval for the same scope.

In [ ]:
import matplotlib.pyplot as plt

holdout = results["predictions"].query(
    "scope == 'complete_164' and evaluation == 'holdout'"
)
future = results["prophet_26_week_forecast"].query("scope == 'complete_164'")

fig, (left, right) = plt.subplots(1, 2, figsize=(15, 4), layout="constrained")
actual = holdout.drop_duplicates("Date")
left.plot(actual["Date"], actual["actual"], label="Actual", color="black", linewidth=2)
for model, rows in holdout.groupby("model", sort=False):
    left.plot(rows["Date"], rows["prediction"], label=model, alpha=0.8)
left.set(title="29-week holdout", ylabel="Weekly sales value")
left.legend(fontsize=8)

right.plot(future["Date"], future["yhat"], label="Prophet")
right.fill_between(future["Date"], future["yhat_lower"], future["yhat_upper"],
                   alpha=0.2, label="Nominal 80% interval")
right.set(title="26-week forecast", ylabel="Weekly sales value")
right.legend()
fig.autofmt_xdate()
plt.show()